# Session 1: Retrieval-Augmented Generation (Student Notebook)

Welcome to Day 3! In this session, you will build production-grade RAG systems. Moving beyond simple keyword matching, you will learn embedding-based retrieval, vector stores, advanced chunking, query transformation, and evaluation — the techniques that make RAG reliable in the real world.

## Learning Objectives

By the end of this session, you will be able to:

1. **Create and compare** text embeddings using OpenAI's embedding models
2. **Build and query** a vector store using ChromaDB
3. **Apply advanced chunking** strategies for better retrieval quality
4. **Transform queries** using HyDE and multi-query techniques
5. **Build an end-to-end RAG pipeline** with source citations
6. **Evaluate RAG quality** with retrieval and generation metrics

In [ ]:
# ============================================================
# Imports and Setup
# ============================================================

import openai
import chromadb
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.documents import Document
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_core.messages import HumanMessage, SystemMessage
import numpy as np
import json
import os

print("All imports successful!")

---
## Demos (Follow Along)

### Demo 1: Embedding Models — Creating and Comparing Text Embeddings

Embeddings convert text into numerical vectors in a high-dimensional space where **semantically similar texts are close together**. This is the foundation of all modern retrieval systems.

In [ ]:
# Demo 1 - Embedding Models

client = openai.OpenAI()

# Step 1: Create embeddings for sample texts
texts = [
    "Python is a popular programming language.",
    "Java is widely used in enterprise software.",
    "Machine learning models can classify images.",
    "Deep learning is a subset of machine learning.",
    "The weather today is sunny and warm."
]

response = client.embeddings.create(
    model="text-embedding-3-small",
    input=texts
)

embeddings = [item.embedding for item in response.data]
print(f"Number of embeddings: {len(embeddings)}")
print(f"Embedding dimensions: {len(embeddings[0])}")
print(f"First 5 values of embedding 0: {embeddings[0][:5]}")

# Step 2: Compute cosine similarity
def cosine_similarity(a, b):
    a, b = np.array(a), np.array(b)
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

print("\nSimilarity matrix:")
print(f"{'':>45}", end="")
for i in range(len(texts)):
    print(f"  [{i}]", end="")
print()
for i, t1 in enumerate(texts):
    print(f"[{i}] {t1[:40]:>42}", end="")
    for j, t2 in enumerate(texts):
        sim = cosine_similarity(embeddings[i], embeddings[j])
        print(f" {sim:.2f}", end="")
    print()

# Step 3: Semantic search
query = "What programming languages are popular?"
query_emb = client.embeddings.create(model="text-embedding-3-small", input=[query]).data[0].embedding

print(f"\nQuery: {query}")
print("Ranked results:")
scored = [(cosine_similarity(query_emb, emb), text) for emb, text in zip(embeddings, texts)]
scored.sort(reverse=True)
for score, text in scored:
    print(f"  {score:.4f} | {text}")

### Demo 2: Vector Stores — Indexing and Similarity Search with ChromaDB

A vector store indexes embeddings for fast similarity search. ChromaDB is a lightweight, in-memory vector store perfect for prototyping.

In [ ]:
# Demo 2 - Vector Stores with ChromaDB

# Step 1: Create a ChromaDB client and collection
chroma_client = chromadb.Client()
collection = chroma_client.create_collection(
    name="demo_docs",
    metadata={"hnsw:space": "cosine"}
)

# Step 2: Add documents with metadata
documents = [
    "LangChain is a framework for building LLM applications with composable components.",
    "LangGraph extends LangChain with graph-based stateful workflows and cycles.",
    "RAG combines retrieval with generation to ground LLM answers in external knowledge.",
    "Vector stores like ChromaDB and FAISS enable fast similarity search over embeddings.",
    "Prompt engineering techniques include few-shot, chain-of-thought, and ReAct patterns.",
    "OpenAI's function calling allows LLMs to invoke external tools with structured arguments.",
    "Pydantic provides data validation and settings management using Python type annotations.",
    "Multi-agent systems use supervisor-worker patterns for complex task decomposition."
]

# Use OpenAI embeddings via the API
client = openai.OpenAI()
emb_response = client.embeddings.create(model="text-embedding-3-small", input=documents)
doc_embeddings = [item.embedding for item in emb_response.data]

collection.add(
    documents=documents,
    embeddings=doc_embeddings,
    ids=[f"doc_{i}" for i in range(len(documents))],
    metadatas=[{"source": f"chapter_{i+1}"} for i in range(len(documents))]
)
print(f"Indexed {collection.count()} documents")

# Step 3: Query the collection
query = "How do I build workflows with LLMs?"
query_emb = client.embeddings.create(model="text-embedding-3-small", input=[query]).data[0].embedding

results = collection.query(
    query_embeddings=[query_emb],
    n_results=3
)

print(f"\nQuery: {query}")
print("Top 3 results:")
for i, (doc, distance, meta) in enumerate(zip(results['documents'][0], results['distances'][0], results['metadatas'][0])):
    print(f"  {i+1}. [{meta['source']}] (dist={distance:.4f}) {doc}")

### Demo 3: Advanced Chunking Strategies

How you split documents directly affects retrieval quality. Different strategies work better for different content types: recursive splitting for general text, sentence-based for conversational content, and semantic chunking for technical documents.

In [ ]:
# Demo 3 - Advanced Chunking Strategies

sample_doc = """# Introduction to Retrieval-Augmented Generation

Retrieval-Augmented Generation (RAG) is a technique that enhances LLM responses by grounding them in external knowledge. The approach was introduced by Lewis et al. in 2020.

## How RAG Works

The RAG pipeline consists of three main stages:

1. Indexing: Documents are split into chunks, embedded, and stored in a vector database.
2. Retrieval: When a user asks a question, the query is embedded and similar chunks are found.
3. Generation: Retrieved chunks are included in the prompt, and the LLM generates an answer.

## Benefits of RAG

RAG reduces hallucinations by providing factual context. It keeps information up-to-date without retraining. It allows source attribution, improving trust and verifiability.

## Challenges

The main challenges include: chunking strategy (too small loses context, too large dilutes relevance), embedding quality, retrieval accuracy, and handling multi-hop questions that require information from multiple chunks."""

# Strategy 1: Fixed-size recursive splitting
recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=200, chunk_overlap=30, separators=["\n\n", "\n", ". ", " "]
)
recursive_chunks = recursive_splitter.split_text(sample_doc)

print(f"Recursive splitting: {len(recursive_chunks)} chunks")
for i, chunk in enumerate(recursive_chunks[:3]):
    print(f"  Chunk {i+1} ({len(chunk)} chars): {chunk[:80]}...")

# Strategy 2: Markdown-aware splitting
md_splitter = RecursiveCharacterTextSplitter(
    chunk_size=300, chunk_overlap=30,
    separators=["\n## ", "\n# ", "\n\n", "\n", ". ", " "]
)
md_chunks = md_splitter.split_text(sample_doc)

print(f"\nMarkdown-aware splitting: {len(md_chunks)} chunks")
for i, chunk in enumerate(md_chunks[:3]):
    print(f"  Chunk {i+1} ({len(chunk)} chars): {chunk[:80]}...")

# Strategy 3: Sentence-level splitting
sentence_splitter = RecursiveCharacterTextSplitter(
    chunk_size=150, chunk_overlap=0,
    separators=[". ", "\n", " "]
)
sentence_chunks = sentence_splitter.split_text(sample_doc)

print(f"\nSentence-level splitting: {len(sentence_chunks)} chunks")
for i, chunk in enumerate(sentence_chunks[:3]):
    print(f"  Chunk {i+1} ({len(chunk)} chars): {chunk[:80]}...")

print(f"\nComparison: Recursive={len(recursive_chunks)}, Markdown={len(md_chunks)}, Sentence={len(sentence_chunks)}")

### Demo 4: Query Transformation Techniques

Users often write queries that are too vague or specific for direct retrieval. Query transformation improves retrieval by rewriting, expanding, or hypothetically answering the query before searching.

In [ ]:
# Demo 4 - Query Transformation

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# Technique 1: Multi-query expansion
def multi_query_expand(question, n=3):
    """Generate multiple alternative queries for better retrieval coverage."""
    response = llm.invoke([
        SystemMessage(content=f"Generate {n} alternative versions of this question. Each should approach it from a different angle. Return as a JSON list of strings."),
        HumanMessage(content=question)
    ])
    try:
        queries = json.loads(response.content)
    except:
        queries = [question]
    return [question] + queries

# Technique 2: HyDE (Hypothetical Document Embeddings)
def hyde_transform(question):
    """Generate a hypothetical answer, then use it for retrieval."""
    response = llm.invoke([
        SystemMessage(content="Write a short paragraph that would be a perfect answer to this question. Do not say 'I think' — write it as if it's from a textbook."),
        HumanMessage(content=question)
    ])
    return response.content

# Test both techniques
question = "How do I make my RAG system more accurate?"

print(f"Original question: {question}")
print("\n--- Multi-Query Expansion ---")
expanded = multi_query_expand(question)
for i, q in enumerate(expanded):
    print(f"  {i+1}. {q}")

print("\n--- HyDE Transform ---")
hypothetical = hyde_transform(question)
print(f"  Hypothetical answer: {hypothetical[:200]}...")
print("\n  (This hypothetical answer would be embedded and used for retrieval")
print("   instead of the original question, often finding more relevant chunks.)")

### Demo 5: End-to-End RAG Pipeline with Source Citations

This demo brings everything together into a complete RAG pipeline that retrieves relevant chunks, generates an answer, and includes source citations.

In [ ]:
# Demo 5 - End-to-End RAG Pipeline with Citations

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
embed_client = openai.OpenAI()

# Step 1: Build knowledge base
kb_docs = [
    {"text": "RAG was introduced by Lewis et al. in 2020. It combines retrieval with generation to reduce hallucinations.", "source": "rag_paper.pdf", "page": 1},
    {"text": "The indexing phase involves chunking documents, creating embeddings, and storing them in a vector database.", "source": "rag_guide.md", "page": 3},
    {"text": "ChromaDB is an open-source embedding database. It supports cosine similarity and Euclidean distance.", "source": "vectordb_docs.md", "page": 1},
    {"text": "Query transformation techniques like HyDE and multi-query improve retrieval by reformulating the question.", "source": "rag_guide.md", "page": 7},
    {"text": "Chunking strategy is critical: too small loses context, too large dilutes relevance. 200-500 tokens is typical.", "source": "best_practices.md", "page": 2},
    {"text": "Reranking uses a cross-encoder to rescore retrieved results, improving precision at the cost of latency.", "source": "rag_guide.md", "page": 9}
]

# Step 2: Index in ChromaDB
chroma = chromadb.Client()
coll = chroma.create_collection(name="rag_demo_e2e", metadata={"hnsw:space": "cosine"})

texts = [d["text"] for d in kb_docs]
embs = [e.embedding for e in embed_client.embeddings.create(model="text-embedding-3-small", input=texts).data]
coll.add(
    documents=texts, embeddings=embs,
    ids=[f"doc_{i}" for i in range(len(kb_docs))],
    metadatas=[{"source": d["source"], "page": d["page"]} for d in kb_docs]
)

# Step 3: RAG function
def rag_query(question, k=3):
    # Retrieve
    q_emb = embed_client.embeddings.create(model="text-embedding-3-small", input=[question]).data[0].embedding
    results = coll.query(query_embeddings=[q_emb], n_results=k)
    
    # Build context with source tags
    context_parts = []
    for doc, meta in zip(results['documents'][0], results['metadatas'][0]):
        context_parts.append(f"[Source: {meta['source']}, p.{meta['page']}] {doc}")
    context = "\n\n".join(context_parts)
    
    # Generate
    response = llm.invoke([
        SystemMessage(content="Answer based on the context. Cite sources in [brackets]. If unsure, say so."),
        HumanMessage(content=f"Context:\n{context}\n\nQuestion: {question}")
    ])
    
    return {"answer": response.content, "sources": results['metadatas'][0], "context": context}

# Step 4: Test
for q in ["What is RAG and who introduced it?", "How should I chunk my documents?", "What is reranking?"]:
    result = rag_query(q)
    print(f"Q: {q}")
    print(f"A: {result['answer']}")
    print(f"Sources: {[s['source'] for s in result['sources']]}\n")

---
## Tasks (Your Turn!)

### Task 1: Build an Embedding-Based Document Search Engine

Create a search engine that takes a corpus of documents, embeds them, and returns the most relevant documents for a query — ranked by cosine similarity.

In [ ]:
# Task 1 - Build an Embedding-Based Document Search Engine

# TODO: Build a SearchEngine class that:
# 1. Takes a list of documents (strings) in __init__
# 2. Embeds all documents using OpenAI embeddings
# 3. Has a search(query, k) method that returns top-k results ranked by similarity
#
# Hint: Use openai.OpenAI().embeddings.create() to embed texts
# Hint: Compute cosine similarity between query embedding and all doc embeddings
# Hint: Return results sorted by similarity score (highest first)

class SearchEngine:
    def __init__(self, documents):
        # YOUR CODE HERE
        pass

    def search(self, query, k=3):
        # YOUR CODE HERE
        pass


# Test
# engine = SearchEngine(["Python is great for ML", "JavaScript runs in browsers", ...])
# results = engine.search("What language for machine learning?")
# for score, doc in results: print(f"{score:.4f} | {doc}")

### Task 2: Implement a Multi-Strategy Chunking Pipeline

Build a chunking pipeline that applies different strategies based on document type (markdown, code, plain text) and evaluates chunk quality.

In [ ]:
# Task 2 - Implement a Multi-Strategy Chunking Pipeline

# TODO: Create a SmartChunker class that:
# 1. Detects document type (markdown, code, plain text)
# 2. Applies appropriate splitting strategy for each type
# 3. Returns chunks with metadata (type, size, overlap)
#
# Hint: Check for "#" headers (markdown), "def "/"class " (code), else plain text
# Hint: Markdown: split on headers; Code: split on function/class boundaries; Plain: recursive
# Hint: Return a list of dicts: {"text": ..., "type": ..., "tokens": ...}

class SmartChunker:
    def __init__(self, chunk_size=300, chunk_overlap=50):
        # YOUR CODE HERE
        pass

    def chunk(self, text):
        # YOUR CODE HERE
        pass


# Test
# chunker = SmartChunker()
# chunks = chunker.chunk("# Introduction\n\nSome text...")
# for c in chunks: print(f"[{c['type']}] {c['text'][:60]}...")

### Task 3: Create a Query Expansion and Reranking System

Build a retrieval system that expands the query into multiple variants, retrieves candidates for each, deduplicates, and reranks using the LLM.

In [ ]:
# Task 3 - Create a Query Expansion and Reranking System

# TODO: Build an AdvancedRetriever class that:
# 1. Expands the query into 3 alternative formulations
# 2. Retrieves candidates for each formulation from a ChromaDB collection
# 3. Deduplicates the combined results
# 4. Reranks using the LLM to score relevance (0-10)
#
# Hint: Use multi_query_expand from Demo 4
# Hint: Use a set to track seen documents for deduplication
# Hint: For reranking, ask the LLM to rate relevance of each doc to the query

class AdvancedRetriever:
    def __init__(self, collection):
        # YOUR CODE HERE
        pass

    def retrieve(self, query, k=3):
        # YOUR CODE HERE
        pass


# Test
# retriever = AdvancedRetriever(collection)
# results = retriever.retrieve("How to improve RAG accuracy?")
# for doc, score in results: print(f"{score}/10 | {doc[:80]}")

### Task 4: Build a Production RAG Pipeline with Evaluation Metrics

Build a complete RAG system that includes automated evaluation — measuring retrieval relevance, answer faithfulness (does the answer match the sources?), and completeness.

In [ ]:
# Task 4 - Build a Production RAG Pipeline with Evaluation

# TODO: Build an EvaluatedRAG class that:
# 1. Has a query() method that returns answer + sources + metrics
# 2. Evaluates retrieval relevance (are the retrieved chunks relevant?)
# 3. Evaluates answer faithfulness (is the answer supported by sources?)
# 4. Evaluates answer completeness (does it fully address the question?)
#
# Hint: Use LLM-as-judge to score each metric 1-5
# Hint: Return metrics as a dict: {"relevance": ..., "faithfulness": ..., "completeness": ...}
# Hint: Average the metrics for an overall quality score

class EvaluatedRAG:
    def __init__(self, documents):
        # YOUR CODE HERE
        pass

    def query(self, question):
        # YOUR CODE HERE
        pass

    def evaluate(self, question, answer, context):
        # YOUR CODE HERE
        pass


# Test
# rag = EvaluatedRAG(documents)
# result = rag.query("What is RAG?")
# print(f"Answer: {result['answer']}")
# print(f"Metrics: {result['metrics']}")

---
## Summary

In this session you learned production-grade RAG engineering:

1. **Embeddings** -- How text is converted to vectors where semantic similarity equals spatial proximity.
2. **Vector Stores** -- How ChromaDB indexes embeddings for sub-millisecond similarity search.
3. **Chunking Strategies** -- How splitting strategy directly affects retrieval quality.
4. **Query Transformation** -- How multi-query and HyDE improve retrieval for complex questions.
5. **End-to-End RAG** -- How to combine retrieval, generation, and source citations into a production pipeline.

**Up Next:** In Session 2, we will learn how to deploy and scale these systems in production.